### 1. Necessary imports and Device configs

In [5]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
!pip install torchinfo

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### 2. Kaggle Dataset Import, CIFAR 10 dataset

### 3. Data Transforms & Subset Partitioning

In [ ]:
import os
import json
import shutil
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

kaggle_credentials = {
    "username": "prince12344",
    "key": "2231f9fc02f78054622a636998475757"
}

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')

with open(kaggle_json_path, 'w') as f:
    json.dump(kaggle_credentials, f)

try:
    os.chmod(kaggle_json_path, 0o600)
except Exception:
    pass

# Also set environment variables for Kaggle API
os.environ['KAGGLE_USERNAME'] = kaggle_credentials['username']
os.environ['KAGGLE_KEY'] = kaggle_credentials['key']

if os.path.exists('/kaggle/input/cifar-10/trainLabels.csv'):
    base_data_dir = '/kaggle/input/cifar-10'
    train_archive = os.path.join(base_data_dir, 'train.7z')
    csv_path = os.path.join(base_data_dir, 'trainLabels.csv')
    !p7zip -d -k {train_archive}
else:
    !pip install -q kaggle py7zr
    if not os.path.exists('cifar10_raw'):
        !kaggle competitions download -c cifar-10
        !unzip -q -o cifar-10.zip -d cifar10_raw
    if not os.path.exists('cifar10_raw/trainLabels.csv'):
        !unzip -q -o cifar10_raw/trainLabels.csv.zip -d cifar10_raw/
    if not os.path.exists('train'):
        try:
            !p7zip -d -k cifar10_raw/train.7z
        except Exception:
            pass
        if not os.path.exists('train'):
            import py7zr
            with py7zr.SevenZipFile('cifar10_raw/train.7z', mode='r') as z:
                z.extractall()
    csv_path = 'cifar10_raw/trainLabels.csv'

# -------------------------------------------------------------
# 3. Class Filtering & Non-Overlapping Dataset Splitting
# -------------------------------------------------------------
CLASS_MAP = {
    'airplane': 0,
    'automobile': 1,
    'bird': 2,
    'cat': 3,
    'dog': 4
}

class KaggleCIFAR5Dataset(Dataset):
    def __init__(self, img_dir, df, transform=None):
        self.img_dir = img_dir
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = f"{row['id']}.png"
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = row['label_idx']

        if self.transform:
            image = self.transform(image)

        return image, label

# Read annotations and filter for target classes
df_labels = pd.read_csv(csv_path)
df_filtered = df_labels[df_labels['label'].isin(CLASS_MAP.keys())].copy()
df_filtered['label_idx'] = df_filtered['label'].map(CLASS_MAP)

# Partition with exact counts per class: 1000 train, 200 val, 200 test
train_rows, val_rows, test_rows = [], [], []

for cls_name, cls_idx in CLASS_MAP.items():
    cls_subset = df_filtered[df_filtered['label_idx'] == cls_idx].sample(
        n=1400, random_state=42
    )
    train_rows.append(cls_subset.iloc[:1000])
    val_rows.append(cls_subset.iloc[1000:1200])
    test_rows.append(cls_subset.iloc[1200:1400])

train_df = pd.concat(train_rows).sample(frac=1, random_state=42).reset_index(drop=True)
val_df = pd.concat(val_rows).sample(frac=1, random_state=42).reset_index(drop=True)
test_df = pd.concat(test_rows).sample(frac=1, random_state=42).reset_index(drop=True)

# -------------------------------------------------------------
# 4. Normalization Transforms & PyTorch DataLoaders
# -------------------------------------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

train_dataset = KaggleCIFAR5Dataset('train', train_df, transform=transform)
val_dataset = KaggleCIFAR5Dataset('train', val_df, transform=transform)
test_dataset = KaggleCIFAR5Dataset('train', test_df, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2 if os.name != 'nt' else 0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2 if os.name != 'nt' else 0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2 if os.name != 'nt' else 0, pin_memory=torch.cuda.is_available())

print(f"Data ready:")
print(f" - Train images: {len(train_dataset)} (1,000 per class)")
print(f" - Val images:   {len(val_dataset)} (200 per class)")
print(f" - Test images:  {len(test_dataset)} (200 per class)")

### 4: Custom CNN Architecture Implementation

In [ ]:
class CustomCIFAR5CNN(nn.Module):
    def __init__(self):
        super(CustomCIFAR5CNN, self).__init__()

        # Conv Block 1 & 2 -> Pool 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv Block 3 & 4 -> Pool 2
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv Block 5 -> Pool 3
        self.conv5 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Classifier Head
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(in_features=128 * 4 * 4, out_features=128)  # 2048 -> 128
        self.dropout = nn.Dropout(p=0.4)
        self.output = nn.Linear(in_features=128, out_features=5)

        self.relu = nn.ReLU()

    def forward(self, x):
        # 3 x 32 x 32 -> 32 x 32 x 32
        x = self.relu(self.conv1(x))
        # 32 x 32 x 32 -> 32 x 32 x 32
        x = self.relu(self.conv2(x))
        # 32 x 32 x 32 -> 32 x 16 x 16
        x = self.pool1(x)

        # 32 x 16 x 16 -> 64 x 16 x 16
        x = self.relu(self.conv3(x))
        # 64 x 16 x 16 -> 64 x 16 x 16
        x = self.relu(self.conv4(x))
        # 64 x 16 x 16 -> 64 x 8 x 8
        x = self.pool2(x)

        # 64 x 8 x 8 -> 128 x 8 x 8
        x = self.relu(self.conv5(x))
        # 128 x 8 x 8 -> 128 x 4 x 4
        x = self.pool3(x)

        # 128 x 4 x 4 -> 2048
        x = self.flatten(x)
        # 2048 -> 128
        x = self.relu(self.fc1(x))
        # Dropout
        x = self.dropout(x)
        # 128 -> 5
        x = self.output(x)
        return x

model = CustomCIFAR5CNN().to(device)

### 5: Model Summary & Layer Shapes Verification

In [ ]:
try:
    from torchinfo import summary
    summary(model, input_size=(1, 3, 32, 32), device=str(device))
except ImportError:
    # Fallback to torchsummary or manual layer dimension inspection
    try:
        from torchsummary import summary
        summary(model, (3, 32, 32))
    except ImportError:
        print(model)
        dummy_input = torch.randn(1, 3, 32, 32).to(device)
        print("\nVerifying output shape with dummy batch:")
        output = model(dummy_input)
        print(f"Output Tensor Shape: {output.shape} (Expected: torch.Size([1, 5]))")

### 6: Training and Validation Routine

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 15

for epoch in range(num_epochs):
    # Training Phase
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    # Validation Phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100.0 * val_correct / val_total

    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

### 7: Model Evaluation & Test Accuracy

In [ ]:
model.eval()
test_loss, test_correct, test_total = 0.0, 0, 0
class_correct = [0] * 5
class_total = [0] * 5
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Dog']

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        test_correct += preds.eq(labels).sum().item()
        test_total += labels.size(0)

        for c in range(labels.size(0)):
            label = labels[c].item()
            pred = preds[c].item()
            if label == pred:
                class_correct[label] += 1
            class_total[label] += 1

print(f"Overall Test Accuracy: {100.0 * test_correct / test_total:.2f}%\n")
print("Per-Class Accuracy:")
for i in range(5):
    acc = 100.0 * class_correct[i] / class_total[i]
    print(f"  {class_names[i]:<10}: {acc:.2f}% ({class_correct[i]}/{class_total[i]})")